# 🤗 Hugging Face Prompt Engineering Playground

Welcome to the **Prompt Engineering Playground**! This notebook is designed to help you experiment with different prompt styles and Hugging Face models using the free **Serverless Inference API** (zero API cost, no local downloads required).

## 🛠️ In this notebook, we will:
1. **Setup the Hugging Face Inference Client**.
2. **Compare Prompt Styles** (Zero-Shot, Role Prompting, Few-Shot, Chain of Thought, Structured Output) side-by-side.
3. **Compare Models** (e.g. Qwen vs Llama vs Mistral) on the same prompt.
4. **Experiment with Parameters** (Temperature) to see how it affects creativity and predictability.
5. **Run an Interactive Chatbot** directly within a Jupyter cell.

### Step 1: Install & Load Dependencies
First, we will load variables from the `.env` file and initialize the `InferenceClient`.

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from IPython.display import display, HTML, Markdown

# Load .env file
load_dotenv()

# Retrieve Hugging Face Token
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    hf_token = hf_token.strip()
if not hf_token or hf_token == "":
    hf_token = None
    print("⚠️ Warning: HF_TOKEN not found in .env. Running without token (strict rate limits may apply).")
else:
    print("✅ Hugging Face token successfully loaded.")

✅ Hugging Face token successfully loaded.


### Step 2: Define the Prompt Techniques
Let's define different prompting techniques for a Sentiment Analysis task.

In [2]:
PROMPTS = {
    "Zero-Shot": 
        "Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'",

    "Role Prompting": 
        "You are a strict Quality Assurance reviewer. Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'",

    "Few-Shot": 
        """Classify sentiment as Positive, Neutral, or Negative.

Review: 'This is the best purchase I have made all year!' -> Sentiment: Positive
Review: 'It works as expected, nothing special.' -> Sentiment: Neutral
Review: 'The product was okay, but customer support was terrible and shipping took two weeks.' -> Sentiment:""",

    "Chain of Thought": 
        """Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'
Let's think step by step:
1. Identify the positive aspects of the review.
2. Identify the negative aspects of the review.
3. Weigh them and determine the final overall sentiment.
4. Output the final sentiment.""",

    "Structured Output (JSON)": 
        """Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'
Return the result strictly in the following JSON format:
{
  "review": "string",
  "sentiment": "Positive/Neutral/Negative",
  "confidence": 0.0 to 1.0,
  "key_issues": ["list of issues found"]
}"""
}

### Step 3: Run and Compare Prompting Styles
We will run the prompts using the default model `Qwen/Qwen2.5-7B-Instruct` and present the results in a beautiful HTML table.

In [3]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
client = InferenceClient(model=model_id, token=hf_token)

html_content = """
<style>
    .comparison-table { width: 100%; border-collapse: collapse; font-family: sans-serif; margin-top: 20px; }
    .comparison-table th, .comparison-table td { border: 1px solid #ddd; padding: 12px; text-align: left; vertical-align: top; }
    .comparison-table th { background-color: #5856d6; color: white; }
    .comparison-table tr:nth-child(even) { background-color: #f9f9f9; }
    .prompt-cell { background-color: #f1f1f7; font-style: italic; font-size: 0.9em; color: #555; }
    .response-cell { white-space: pre-wrap; font-family: monospace; font-size: 0.95em; }
</style>
<table class='comparison-table'>
    <tr>
        <th style='width: 20%'>Technique</th>
        <th style='width: 35%'>Prompt Template</th>
        <th style='width: 45%'>Model Output</th>
    </tr>
"""

print(f"Querying Qwen/Qwen2.5-7B-Instruct serverless API...")
for style, prompt_text in PROMPTS.items():
    clean_prompt = prompt_text.strip().replace("\n", "<br>").replace("<", "&lt;").replace(">", "&gt;")
    try:
        response = client.chat_completion(messages=[{"role": "user", "content": prompt_text}], max_tokens=200, temperature=0.1)
        clean_response = response.choices[0].message.content.strip().replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        
        html_content += f"""
        <tr>
            <td><b>{style}</b></td>
            <td class='prompt-cell'>{clean_prompt}</td>
            <td class='response-cell'>{clean_response}</td>
        </tr>
        """
    except Exception as e:
        html_content += f"""
        <tr>
            <td><b>{style}</b></td>
            <td class='prompt-cell'>{clean_prompt}</td>
            <td style='color: red;'>Error: {str(e)}</td>
        </tr>
        """

html_content += "</table>"
display(HTML(html_content))

Querying Qwen/Qwen2.5-7B-Instruct serverless API...


Technique,Prompt Template,Model Output
Zero-Shot,"Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'","The sentiment of this review can be classified as mixed, but leaning towards negative. The review acknowledges that the product itself was acceptable (""okay""), which is a neutral to slightly positive point. However, the customer support is described as ""terrible,"" and the shipping time is criticized for being too long (""took two weeks""), which are both negative aspects. The negative points seem to outweigh the neutral one, giving the overall sentiment a negative leaning."
Role Prompting,"You are a strict Quality Assurance reviewer. Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'","The sentiment of this review can be classified as negative. The review indicates dissatisfaction with the customer support and shipping time, despite the product being merely satisfactory."
Few-Shot,"Classify sentiment as Positive, Neutral, or Negative.<br><br>Review: 'This is the best purchase I have made all year!' -> Sentiment: Positive<br>Review: 'It works as expected, nothing special.' -> Sentiment: Neutral<br>Review: 'The product was okay, but customer support was terrible and shipping took two weeks.' -> Sentiment:","Sentiment: NegativeThe review expresses dissatisfaction with the customer support and the shipping time, which outweighs the neutral statement about the product being ""okay."""
Chain of Thought,"Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'<br>Let's think step by step:<br>1. Identify the positive aspects of the review.<br>2. Identify the negative aspects of the review.<br>3. Weigh them and determine the final overall sentiment.<br>4. Output the final sentiment.","Let's break down the review step by step:1. **Identify the positive aspects of the review:** - ""The product was okay"" suggests that the product met basic expectations but did not exceed them. This is a neutral to slightly positive statement.2. **Identify the negative aspects of the review:** - ""Customer support was terrible"" indicates a significant negative experience with the company's support team. - ""Shipping took two weeks"" suggests that the delivery time was longer than expected, which is another negative point.3. **Weigh them and determine the final overall sentiment:** - The positive aspect is mild and neutral. - The negative aspects are more significant and impactful. The poor customer support and long shipping time are likely to have a more lasting negative impression on the customer.4. **Output the final sentiment:** - Given that the negative aspects outweigh the positive one, the overall sentiment of the review is negative."
Structured Output (JSON),"Classify the sentiment of this review: 'The product was okay, but customer support was terrible and shipping took two weeks.'<br>Return the result strictly in the following JSON format:<br>{<br> ""review"": ""string"",<br> ""sentiment"": ""Positive/Neutral/Negative"",<br> ""confidence"": 0.0 to 1.0,<br> ""key_issues"": [""list of issues found""]<br>}","{ ""review"": ""The product was okay, but customer support was terrible and shipping took two weeks."", ""sentiment"": ""Negative"", ""confidence"": 0.8, ""key_issues"": [""terrible customer support"", ""long shipping time""]}"


### Step 4: Compare Models on the Same Prompt
Now let's compare three different models using the same structured query to see how their reasoning and formatting differ.

In [4]:
target_prompt = PROMPTS["Structured Output (JSON)"]
compare_models = [
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct"
]

html_models = """
<style>
    .model-table { width: 100%; border-collapse: collapse; font-family: sans-serif; }
    .model-table th, .model-table td { border: 1px solid #ddd; padding: 12px; text-align: left; vertical-align: top; }
    .model-table th { background-color: #007aff; color: white; }
    .model-table tr:nth-child(even) { background-color: #f9f9f9; }
</style>
<table class='model-table'>
    <tr>
        <th style='width: 30%'>Model</th>
        <th style='width: 70%'>Output</th>
    </tr>
"""

for m_id in compare_models:
    print(f"Querying {m_id}...")
    try:
        m_client = InferenceClient(model=m_id, token=hf_token)
        res = m_client.chat_completion(messages=[{"role": "user", "content": target_prompt}], max_tokens=256, temperature=0.1)
        clean_res = res.choices[0].message.content.strip().replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        
        html_models += f"""
        <tr>
            <td><b>{m_id}</b></td>
            <td style='font-family: monospace; font-size: 0.95em;'>{clean_res}</td>
        </tr>
        """
    except Exception as e:
        html_models += f"""
        <tr>
            <td><b>{m_id}</b></td>
            <td style='color: red;'>Error: {str(e)}</td>
        </tr>
        """

html_models += "</table>"
display(HTML(html_models))

Querying Qwen/Qwen2.5-7B-Instruct...
Querying microsoft/Phi-3-mini-4k-instruct...
Querying Qwen/Qwen2.5-Coder-7B-Instruct...


Model,Output
Qwen/Qwen2.5-7B-Instruct,"{ ""review"": ""The product was okay, but customer support was terrible and shipping took two weeks."", ""sentiment"": ""Negative"", ""confidence"": 0.8, ""key_issues"": [""terrible customer support"", ""long shipping time""]}"
microsoft/Phi-3-mini-4k-instruct,"Error: (Request ID: Root=1-6a29a28c-2db8a2b05bc9788843386b5d;58080b74-1020-4cad-9d30-cd0153720490) Bad request: {'message': ""The requested model 'microsoft/Phi-3-mini-4k-instruct' is not supported by any provider you have enabled."", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}"
Qwen/Qwen2.5-Coder-7B-Instruct,"```json{ ""review"": ""The product was okay, but customer support was terrible and shipping took two weeks."", ""sentiment"": ""Negative"", ""confidence"": 0.8, ""key_issues"": [ ""terrible customer support"", ""shipping took two weeks"" ]}```"


### Step 5: Understanding Temperature
Temperature controls the randomness/creativity of the generation.
- **Low temperature (e.g. 0.1)** makes the model deterministic, focusing on the most likely tokens.
- **High temperature (e.g. 1.2)** makes the model highly creative but prone to inconsistencies or hallucinations.

Let's ask the model to generate a **3-word slogan** for a coffee brand 3 times using low temperature vs high temperature.

In [5]:
slogan_prompt = "Write a creative 3-word slogan for a coffee brand. Output ONLY the 3 words."
coffee_client = InferenceClient(model="Qwen/Qwen2.5-7B-Instruct", token=hf_token)

print("--- LOW TEMPERATURE (0.1) ---")
for i in range(3):
    out = coffee_client.chat_completion(messages=[{"role": "user", "content": slogan_prompt}], max_tokens=20, temperature=0.1)
    print(f"Run {i+1}: {out.choices[0].message.content.strip()}")

print("\n--- HIGH TEMPERATURE (1.3) ---")
for i in range(3):
    out = coffee_client.chat_completion(messages=[{"role": "user", "content": slogan_prompt}], max_tokens=20, temperature=1.3)
    print(f"Run {i+1}: {out.choices[0].message.content.strip()}")

--- LOW TEMPERATURE (0.1) ---
Run 1: Elevate Every Moment
Run 2: Elevate Every Moment
Run 3: Elevate Every Moment

--- HIGH TEMPERATURE (1.3) ---
Run 1: ELEMENTS OF BRI策EAU
Run 2: Eclfysiỏ
Run 3: Wakeful Moments


### Step 6: Interactive Chatbot inside Jupyter
Run this cell to start an interactive chatbot session directly inside this notebook. You can converse with the model, adjust its system prompt, and test your own prompts dynamically!

In [ ]:
# Initialize Chatbot client
chat_model_id = "Qwen/Qwen2.5-7B-Instruct"
chat_client = InferenceClient(model=chat_model_id, token=hf_token)

system_instruction = input("Enter System Prompt (Default: 'You are a helpful and concise AI assistant.'): ").strip()
if not system_instruction:
    system_instruction = "You are a helpful and concise AI assistant."

chat_history = [{"role": "system", "content": system_instruction}]

print(f"\n🤖 Chatbot started using '{chat_model_id}'!")
print("Type 'exit' or 'quit' to stop. Type 'reset' to clear conversation history.\n")

while True:
    try:
        user_msg = input("You: ")
        if user_msg.strip().lower() in ["exit", "quit"]:
            print("Goodbye!")
            break
        if user_msg.strip().lower() == "reset":
            chat_history = [{"role": "system", "content": system_instruction}]
            print("System: Chat history has been reset.\n")
            continue
        if not user_msg.strip():
            continue
            
        chat_history.append({"role": "user", "content": user_msg})
        
        # Get completion
        response = chat_client.chat_completion(
            messages=chat_history,
            max_tokens=512,
            temperature=0.7
        )
        
        reply = response.choices[0].message.content.strip()
        print(f"AI: {reply}\n")
        
        chat_history.append({"role": "assistant", "content": reply})
        
    except KeyboardInterrupt:
        print("\nSession ended by interrupt.")
        break
    except Exception as e:
        print(f"Error: {e}\n")


🤖 Chatbot started using 'Qwen/Qwen2.5-7B-Instruct'!
Type 'exit' or 'quit' to stop. Type 'reset' to clear conversation history.

AI: Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?

